# Notebook 1: Drug 2D Feature Extraction (MG-BERT)

**Purpose:** Extract 2D topological drug features using the pre-trained MG-BERT model.

**Input:** `dataset/drugbank_dataset/metadata.csv` — DrugBank drug-target pairs with SMILES strings.

**Output:** `drug_embeddings_pooled.npy` — a dictionary mapping each drug SMILES to a 256-dimensional feature vector (mean-pooled over atoms).

**Method:** MG-BERT encodes each drug molecule as an atom-level feature matrix of shape `(N_atoms, 256)`.
Mean pooling is applied across the atom dimension to obtain a fixed-size representation per drug.

## Step 0: Mount Google Drive and Extract Project Files

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

# Path to the project zip file on Google Drive
zip_path = '/content/drive/MyDrive/Capstone_Project/EviDTI-main.zip'

# Extract to Colab local directory
!unzip -q "$zip_path" -d /content/

# Change working directory to the extracted folder
os.chdir('/content/EviDTI-main')

# Verify files are present
!ls

## Step 1: Install Dependencies

In [ ]:
# Install RDKit
!pip install rdkit

# Install PyG
!pip install torch_geometric

# Check environment versions
import numpy
import pandas
import torch
print(f"Numpy version: {numpy.__version__}")
print(f"Pandas version: {pandas.__version__}")
print(f"PyTorch version: {torch.__version__}")
!pip install yacs

## Step 2: Download MG-BERT Dependencies

The MG-BERT model requires three helper files from the
[MG-BERT repository](https://github.com/zhang-xuan1314/Molecular-graph-BERT):
`model.py`, `dataset.py` (saved as `dataset_dti.py`), and `utils.py` (saved as `utils_graph.py`).

In [ ]:
import requests

# Download model.py from the MG-BERT repository
url = "https://raw.githubusercontent.com/zhang-xuan1314/Molecular-graph-BERT/main/model.py"

try:
    r = requests.get(url)
    if r.status_code == 200:
        with open('model.py', 'wb') as f:
            f.write(r.content)
        print("model.py downloaded.")
    else:
        print("Download failed.")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
import requests

# Download dataset.py from the MG-BERT repository
url = "https://raw.githubusercontent.com/zhang-xuan1314/Molecular-graph-BERT/main/dataset.py"

print("Downloading dataset_dti.py...")

try:
    r = requests.get(url)
    if r.status_code == 200:
        # Save as dataset_dti.py
        with open('dataset_dti.py', 'wb') as f:
            f.write(r.content)
        print("dataset_dti.py downloaded.")
        
    else:
        print(f"Download failed, status code: {r.status_code}")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
import requests

# Step 1: Download utils.py from MG-BERT and save as utils_graph.py

url = "https://raw.githubusercontent.com/zhang-xuan1314/Molecular-graph-BERT/main/utils.py"
print("Downloading utils_graph.py...")

try:
    r = requests.get(url)
    if r.status_code == 200:
        with open('utils_graph.py', 'wb') as f:
            f.write(r.content)
        print("utils_graph.py downloaded.")
    else:
        print(f"Download failed, status code: {r.status_code}")
except Exception as e:
    print(f"Error: {e}")

# Step 2: Update import path in dataset_dti.py

print("Updating import path in dataset_dti.py...")

try:
    # Read dataset_dti.py
    with open('dataset_dti.py', 'r') as f:
        lines = f.readlines()

    # Replace import statement
    new_lines = []
    modified = False
    for line in lines:
        if 'from utils import smiles2adjoin' in line:
            new_lines.append(line.replace('from utils', 'from utils_graph'))
            modified = True
        else:
            new_lines.append(line)

    # Write back if modified
    if modified:
        with open('dataset_dti.py', 'w') as f:
            f.writelines(new_lines)
        print("dataset_dti.py import path updated.")
    else:
        print("No changes needed.")

except Exception as e:
    print(f"Error: {e}")

In [ ]:
# Linux-compiled version required for Colab
!pip install openbabel-wheel

## Step 3: Apply Patches

**Patch 3a (`model.py`):** Fix positional argument passing for TF2 compatibility.

**Patch 3b (`extract_drug_2d_emb.py`, `metadata.csv`):** Generate the deduplicated drug list
(`unique_drugs.csv`) and update the extraction script to use it.

In [ ]:
import os

print("Fixing model.py parameter passing for TF2 compatibility...")

target_file = "/content/EviDTI-main/model.py"

if os.path.exists(target_file):
    # Read file
    with open(target_file, "r") as f:
        lines = f.readlines()

    # Scan and fix
    new_lines = []
    fixed_count = 0
    for line in lines:
        if "self.enc_layers[i](x, training, mask, adjoin_matrix)" in line:
            new_line = line.replace(
                "self.enc_layers[i](x, training, mask, adjoin_matrix)",
                "self.enc_layers[i](x, training=training, mask=mask, adjoin_matrix=adjoin_matrix)"
            )
            new_lines.append(new_line)
            fixed_count += 1

        else:
            new_lines.append(line)

    # Write back
    if fixed_count > 0:
        with open(target_file, "w") as f:
            f.writelines(new_lines)
        print(f"model.py fixed ({fixed_count} location(s)).")

    else:
        print("Target line not found, may already be fixed.")


else:
    print("model.py not found.")

In [ ]:
import os
import pandas as pd
import re

print("Generating unique drug list and updating extraction script...")

# Step 1: Generate deduplicated drug list (unique_drugs.csv)
source_path = "/content/EviDTI-main/dataset/drugbank_dataset/metadata.csv"
unique_drug_path = "data/clf/unique_drugs.csv"

if os.path.exists(source_path):
    print(f"Reading source data: {source_path}")
    df = pd.read_csv(source_path)

    print(f"Total pairs: {len(df)}")

    # Deduplicate on SMILES column so each drug appears only once

    target_col = 'smile'
    id_col = 'cid'

    if target_col not in df.columns:  # fallback column name search
        for c in df.columns:
            if 'smi' in c.lower(): target_col = c

    # Deduplicate
    unique_df = df.drop_duplicates(subset=[target_col]).copy()

    # Sort by ID for reproducibility
    if id_col in unique_df.columns:
        unique_df = unique_df.sort_values(by=id_col)

    # Add Label column required by the extraction script
    unique_df['Label'] = 1
    # Add uppercase SMILES column as required by the extraction script
    unique_df['SMILES'] = unique_df[target_col]

    print(f"Unique drugs: {len(unique_df)}")


    # Save
    os.makedirs(os.path.dirname(unique_drug_path), exist_ok=True)
    unique_df.to_csv(unique_drug_path, index=False)
    print(f"Saved to: {unique_drug_path}")

    # Update extract_drug_2d_emb.py to point to unique_drugs.csv
    script_path = "extract_drug_2d_emb.py"
    with open(script_path, "r") as f:
        code = f.read()


    pattern_path = r"Graph_Classification_Dataset\s*\(\s*r?['\"].*?['\"]"
    if re.search(pattern_path, code):
        code = re.sub(pattern_path, f"Graph_Classification_Dataset(r'{os.path.abspath(unique_drug_path)}'", code)
    else:
        # Fallback: direct replacement
        code = code.replace(source_path, os.path.abspath(unique_drug_path))
        if "ready_to_go.csv" in code:
             code = code.replace("data/clf/ready_to_go.csv", unique_drug_path)

    # Ensure smiles_field points to the SMILES column we just created
    if "smiles_field='SMILES'" not in code:
        # Simple replacement
        code = re.sub(r"smiles_field=['\"].*?['\"]", "smiles_field='SMILES'", code)

    with open(script_path, "w") as f:
        f.write(code)
    print("Extraction script updated.")

    # Double-check model.py is fixed
    model_path = "/content/EviDTI-main/model.py"
    if os.path.exists(model_path):
        with open(model_path, "r") as f:
            model_code = f.read()

        # Check if fix is still needed
        if "self.enc_layers[i](x, training, mask, adjoin_matrix)" in model_code:
            model_code = model_code.replace(
                "self.enc_layers[i](x, training, mask, adjoin_matrix)",
                "self.enc_layers[i](x, training=training, mask=mask, adjoin_matrix=adjoin_matrix)"
            )
            with open(model_path, "w") as f:
                f.write(model_code)
            print("model.py fixed.")
        else:
            print("model.py already fixed.")

    print("All preparations complete.")


else:
    print("metadata.csv not found.")

## Step 4: Rewrite Extraction Script and Fix Model Config

Overwrites `extract_drug_2d_emb.py` with a corrected version that fixes model architecture
to Medium config (6 layers, 256-dim) and loads pre-trained MG-BERT weights correctly.

Additional patches are applied directly to `model.py` to override hardcoded Small-config
defaults, and the weight loader is updated to use `by_name=True, skip_mismatch=True`
so that pre-trained weights load correctly regardless of layer count mismatches.

In [ ]:
import os
import re

print("Rewriting extract_drug_2d_emb.py with the corrected final version...")

# Fix model.py (ensure TF2 compatibility)
model_path = "/content/EviDTI-main/model.py"
if os.path.exists(model_path):
    with open(model_path, "r", encoding="utf-8") as f:
        content = f.read()
    content = re.sub(r"\(x,\s*training,\s*mask,\s*adjoin_matrix\)", "(x, training=training, mask=mask, adjoin_matrix=adjoin_matrix)", content)
    with open(model_path, "w", encoding="utf-8") as f:
        f.write(content)
    print("model.py compatibility patch applied.")

# Overwrite extract_drug_2d_emb.py with the corrected final version
file_path = "/content/EviDTI-main/extract_drug_2d_emb.py"


perfect_code = """import tensorflow as tf
import tensorflow.keras as keras
import pandas as pd
import numpy as np
import os
from dataset_dti import Graph_Classification_Dataset
from model import PredictModel, BertModel

os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
keras.backend.clear_session()
os.environ['CUDA_VISIBLE_DEVICES'] = "0"

def main(seed):
    task = 'HY_QS_64_drug_info'
    print(f"Running task: {task}")

    arch = {'name':'Medium', 'path':'medium_weights'}
    trained_epoch = 10

    np.random.seed(seed)
    tf.random.set_seed(seed)

    # Fix vocabulary size
    vocab_size = 65

    dataset_path = f'data/clf/{task}.csv'
    train_dataset, test_dataset, val_dataset = Graph_Classification_Dataset(dataset_path, smiles_field='SMILES', label_field='Label', addH=True).get_data()

    x, adjoin_matrix, y = next(iter(train_dataset.take(1)))
    seq = tf.cast(tf.math.equal(x, 0), tf.float32)
    mask = seq[:, tf.newaxis, tf.newaxis, :]

    print("Initialising PredictModel (6 layers, Medium config)...")
    # Fix model layers to Medium config (6 layers)
    model = PredictModel(num_layers=6, d_model=256, dff=512, num_heads=8, vocab_size=vocab_size, dense_dropout=0.5)

    print("Initialising BertModel (6 layers, Medium config)...")
    temp = BertModel(num_layers=6, d_model=256, dff=512, num_heads=8, vocab_size=vocab_size)

    pred = temp(x, mask=mask, training=True, adjoin_matrix=adjoin_matrix)

    weight_path = arch['path']+'/bert_weights{}_{}.h5'.format(arch['name'],trained_epoch)
    print(f"Loading pre-trained weights: {weight_path}")
    temp.load_weights(weight_path)

    encoder_path = arch['path']+'/bert_weights_encoder{}_{}.h5'.format(arch['name'],trained_epoch)
    temp.encoder.save_weights(encoder_path)
    del temp

    pred = model(x,mask=mask,training=True,adjoin_matrix=adjoin_matrix)
    model.encoder.load_weights(encoder_path)
    print("Pre-trained weights loaded.")

    drug_feature_dic = {}

    id_df = pd.read_csv(dataset_path)
    id_list = id_df['ID'] if 'ID' in id_df.columns else id_df['SMILES']

    print(f"Extracting features for {len(id_list)} drugs...")

    index = 0
    # Iterate over all splits to extract features for every drug
    for dset in [train_dataset, val_dataset, test_dataset]:
        for x, adjoin_matrix, y in dset:
            seq = tf.cast(tf.math.equal(x, 0), tf.float32)
            mask = seq[:, tf.newaxis, tf.newaxis, :]

            # Extract features
            preds = model.encoder(x, mask=mask, training=False, adjoin_matrix=adjoin_matrix)

            # Convert to numpy immediately to avoid GPU memory accumulation
            for p in preds:
                if index < len(id_list):

                    drug_feature_dic[id_list.values[index]] = p.numpy()
                    index += 1

            # Progress update
            if index % 500 == 0:
                print(f"Progress: {index} / {len(id_list)}")

    # Save features
    np.save('drug_embeddings.npy', drug_feature_dic)
    print(f"Done. Features saved to drug_embeddings.npy ({len(drug_feature_dic)} drugs).")

if __name__ == '__main__':
    main(5)
"""

with open(file_path, "w", encoding="utf-8") as f:
    f.write(perfect_code)

print("extract_drug_2d_emb.py successfully rewritten.")

In [ ]:
import os

# Update task name to match unique_drugs.csv
file_path = "/content/EviDTI-main/extract_drug_2d_emb.py"
if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Update task name to match unique_drugs.csv
    content = content.replace("task = 'HY_QS_64_drug_info'", "task = 'unique_drugs'")

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)
    print("Task name updated to: unique_drugs")
else:
    print("extract_drug_2d_emb.py not found.")




## Step 5: Run Feature Extraction

In [ ]:
import os
import re

print("Patching model.py default layer config to Medium (6 layers, 256-dim, 8 heads)...")
file_path = "/content/EviDTI-main/model.py"

if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Fix default num_layers from 3 (Small) to 6 (Medium)
    new_content = re.sub(r"num_layers\s*=\s*3", "num_layers=6", content)
    new_content = re.sub(r"n_layers\s*=\s*3", "n_layers=6", new_content)

    # Also fix self.num_layers if hardcoded
    new_content = re.sub(r"self\.num_layers\s*=\s*3", "self.num_layers = 6", new_content)

    # Fix d_model from 128 (Small) to 256 (Medium)
    new_content = re.sub(r"d_model\s*=\s*128", "d_model=256", new_content)

    # Fix num_heads from 4 (Small) to 8 (Medium)
    new_content = re.sub(r"num_heads\s*=\s*4", "num_heads=8", new_content)

    if new_content != content:
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(new_content)
        print("model.py default config updated to Medium: 6 layers / 256-dim / 8 heads.")

    else:
        print("No default values found to replace, may already be patched.")
else:
    print("model.py not found.")

In [ ]:
import os

print("Patching weight loader to use by_name=True, skip_mismatch=True...")
file_path = "/content/EviDTI-main/extract_drug_2d_emb.py"

if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Load weights by layer name and skip mismatched layers
    # This allows the pre-trained weights to load even if layer counts differ
    if "temp.load_weights(weight_path)" in content:
        content = content.replace(
            "temp.load_weights(weight_path)",
            "temp.load_weights(weight_path, by_name=True, skip_mismatch=True)"
        )
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(content)
        print("Weight loader patched: by_name=True, skip_mismatch=True.")
    else:
        print("Load statement not found, may already be patched.")
else:
    print("extract_drug_2d_emb.py not found.")

In [ ]:
# Run the 2D drug feature extraction script
%cd /content/EviDTI-main
!python extract_drug_2d_emb.py

## Step 6: Mean Pooling

The extraction script outputs atom-level features of shape `(N_atoms, 256)` per drug.
Mean pooling is applied to produce a single fixed-size vector `(256,)` per drug.

In [ ]:
import numpy as np
import os

print("Applying mean pooling to compress atom-level features...")

# Load raw atom-level features: {drug_id -> (N_atoms, 256)}
drug_full_path = '/content/EviDTI-main/drug_embeddings.npy'
drug_full = np.load(drug_full_path, allow_pickle=True).item()

# Apply mean pooling: (N_atoms, 256) -> (256,)
drug_pooled = {}
for drug_id, feat_matrix in drug_full.items():
    # Handle edge case where feature is already 1D
    if len(feat_matrix.shape) > 1:
        # Mean pool over atom dimension
        drug_pooled[drug_id] = np.mean(feat_matrix, axis=0).astype(np.float32)
    else:
        drug_pooled[drug_id] = feat_matrix.astype(np.float32)

# Save pooled features
pooled_path = '/content/EviDTI-main/drug_embeddings_pooled.npy'
np.save(pooled_path, drug_pooled)

# Verify output
size_mb = os.path.getsize(pooled_path) / (1024**2)
sample_shape = list(drug_pooled.values())[0].shape


print("Mean pooling complete.")

print(f"File size: {size_mb:.2f} MB")
print(f"Feature dimension: {sample_shape}  (expected: (256,))")

## Step 7: Save Results to Google Drive

In [ ]:
from google.colab import drive, files
import shutil
import datetime
import os

print("Backing up project to Google Drive...")
drive.mount('/content/drive')

# Key output file paths
protein_path = "/content/EviDTI-main/dataset/bisai/protein_features_pooled.npy"
drug_path = "/content/EviDTI-main/drug_embeddings_pooled.npy"
csv_path = "/content/EviDTI-main/data/clf/unique_drugs.csv"
source_dir = "/content/EviDTI-main"



# Verify key output files exist
if os.path.exists(protein_path) and os.path.exists(drug_path):

    p_size_mb = os.path.getsize(protein_path) / (1024**2)
    d_size_mb = os.path.getsize(drug_path) / (1024**2)

    print(f"Protein features: {p_size_mb:.2f} MB")
    print(f"Drug 2D features: {d_size_mb:.2f} MB")

    # Pack and upload to Google Drive
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
    backup_name = f"EviDTI_Protein_and_Drug_2D_Feature_Backup_{timestamp}"
    destination_zip = f"/content/drive/MyDrive/{backup_name}"

    
    print("Packing project...")

    shutil.make_archive(destination_zip, 'zip', source_dir)
    print(f"Backup saved: {backup_name}.zip")